# TractionMachinePM2D — Process Chain Showcase

This notebook walks through the complete parametric FEM toolchain for a
**48-slot / 8-pole / 3-phase inset-PM traction motor**:

1. Motor design parameters (`motor_params.py`)
2. Full-machine schematic
3. Stator sector geometry & mesh (`gen_stator.py`)
4. Rotor sector geometry & mesh (`gen_rotor.py`)
5. Combined sector mesh & airgap detail (`gen_mesh.py`)
6. Phase / winding assignment
7. FEA model generation (`gen_sif.py`)
8. Postprocessing results (`postprocess.py`)

In [ ]:
import math, sys, re, tempfile
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import matplotlib.patches as mpatches
from matplotlib.patches import Wedge
import matplotlib.colors as mcolors
import gmsh

sys.path.insert(0, str(Path('.').resolve()))
from motor_params import MotorParams
from gen_stator import build_stator
from gen_rotor import build_rotor
from gen_mesh import build_mesh
from gen_sif import gen_sif, _PHASE_MAP
from postprocess import (
    read_scalars, read_losses, summarise, print_summary,
    stator_body_id, rotor_body_id,
)

π = math.pi
RESULTS_DIR = Path('results')

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
})

# ── Phase and body colours ────────────────────────────────────────────────────
PHASE_COLORS = {
    'A+': '#E84040', 'A-': '#C00000',
    'B+': '#3AB03A', 'B-': '#007000',
    'C+': '#AA44FF', 'C-': '#7700CC',
}
IRON_COLOR   = '#4472C4'
AIRGAP_COLOR = '#C8E4F8'
MAG_COLOR    = '#CC44CC'
POCKET_COLOR = '#AAAAEE'
SHAFT_COLOR  = '#AAAAAA'
OPEN_COLOR   = '#FFF8C8'
INSUL_COLOR  = '#FFB060'


def body_color(name):
    if 'Stator_Iron'  in name: return IRON_COLOR
    if 'Rotor_Iron'   in name: return '#5A7FBF'
    if 'Airgap'       in name: return AIRGAP_COLOR
    if 'Magnet'       in name: return MAG_COLOR
    if 'AirPocket'    in name: return POCKET_COLOR
    if 'Shaft'        in name: return SHAFT_COLOR
    if 'Opening'      in name: return OPEN_COLOR
    if 'Insul'        in name: return INSUL_COLOR
    for ph, c in PHASE_COLORS.items():
        if name.endswith(ph): return c
    return '#DDDDDD'


def load_msh(msh_path):
    """Load .msh4; return (xy_mm, tris{name: (N,3)}, segs{name: (M,2)})."""
    if gmsh.isInitialized():
        gmsh.finalize()
    gmsh.initialize()
    gmsh.option.setNumber('General.Terminal', 0)
    gmsh.model.add('viz')
    try:
        gmsh.merge(str(msh_path))
        ntags, coords, _ = gmsh.model.mesh.getNodes()
        xy = coords.reshape(-1, 3)[:, :2]          # mm (model units)
        t2i = {int(t): i for i, t in enumerate(ntags)}
        tris, segs = {}, {}
        for dim, pg in gmsh.model.getPhysicalGroups():
            nm  = gmsh.model.getPhysicalName(dim, pg)
            ens = gmsh.model.getEntitiesForPhysicalGroup(dim, pg)
            if dim == 2:
                rows = []
                for e in ens:
                    etypes, _, ntags_e = gmsh.model.mesh.getElements(2, e)
                    for etype, nn in zip(etypes, ntags_e):
                        if int(etype) == 2:
                            rows.append(nn.reshape(-1, 3))
                if rows:
                    raw = np.vstack(rows)
                    tris[nm] = np.array([[t2i[int(x)] for x in r] for r in raw])
            elif dim == 1:
                rows = []
                for e in ens:
                    etypes, _, ntags_e = gmsh.model.mesh.getElements(1, e)
                    for etype, nn in zip(etypes, ntags_e):
                        if int(etype) == 1:
                            rows.append(nn.reshape(-1, 2))
                if rows:
                    raw = np.vstack(rows)
                    segs[nm] = np.array([[t2i[int(x)] for x in r] for r in raw])
        return xy, tris, segs
    finally:
        gmsh.finalize()


def fill_tris(ax, xy, tris, color_fn=body_color, alpha=1.0, lw=0.12):
    """Draw filled triangles from tris dict."""
    for name, tri in tris.items():
        if len(tri) == 0:
            continue
        c = color_fn(name)
        triang = mtri.Triangulation(xy[:, 0], xy[:, 1], tri)
        ax.tripcolor(triang, facecolors=np.zeros(len(tri)),
                     cmap=mcolors.ListedColormap([c]), alpha=alpha)
        ax.triplot(triang, color='#333333', lw=lw, alpha=0.35)


print('Imports OK')

## 1. Motor Design Parameters

In [ ]:
p = MotorParams()
print(p)

# Quick sanity check
p.validate()
print(f'\nValidation passed. SCALE = {p.SCALE:.4f} m  '
      f'(SCALE = Qp × L_active = {p.Qp} × {p.L_active*1e-3:.3f} m)')

## 2. Full-Machine Cross-Section Schematic

The 360° schematic is constructed analytically from `MotorParams` — no meshing required.
Coloured wedges show the winding layout for all 48 slots.
The dashed circle is the **sliding boundary** (SB) at the mid-airgap radius.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect('equal')
ax.axis('off')

# ── Full winding pattern (8 sectors, alternating polarity) ────────────────────
full_phases = []
for sector in range(p.Qp):
    sign = +1 if sector % 2 == 0 else -1
    for ph in _PHASE_MAP:
        if sign == -1:
            ph = ph[0] + ('-' if ph[1] == '+' else '+')
        full_phases.append(ph)

slot_deg = 360.0 / p.Qs                              # slot pitch in degrees
slot_centres = [(i + 0.5) * slot_deg for i in range(p.Qs)]

r_slot_mid = p.R_si + p.h1 + p.h_slot / 2
slot_half_deg = math.degrees(math.asin(min(p.b_slot / 2 / r_slot_mid, 0.9999)))
open_half_deg = math.degrees(math.asin(min(p.b1   / 2 / p.R_si,       0.9999)))

# ── Painter's algorithm (back to front) ──────────────────────────────────────
# 1. Stator iron disk
ax.add_patch(plt.Circle((0, 0), p.R_so, color=IRON_COLOR, zorder=1))

# 2. Slot winding bodies (phase coloured)
for c_deg, ph in zip(slot_centres, full_phases):
    wedge = Wedge((0, 0), p.R_si + p.h1 + p.h_slot,
                  c_deg - slot_half_deg, c_deg + slot_half_deg,
                  width=p.h_slot, color=PHASE_COLORS[ph], zorder=2)
    ax.add_patch(wedge)

# 3. Re-cover tooth tips with iron (draws the tooth-tip ring)
ax.add_patch(plt.Circle((0, 0), p.R_si + p.h1, color=IRON_COLOR, zorder=3))

# 4. Slot openings (narrow, at bore level)
for c_deg in slot_centres:
    wedge = Wedge((0, 0), p.R_si + p.h1,
                  c_deg - open_half_deg, c_deg + open_half_deg,
                  width=p.h1, color=OPEN_COLOR, zorder=4)
    ax.add_patch(wedge)

# 5. Airgap ring
ax.add_patch(plt.Circle((0, 0), p.R_si, color=AIRGAP_COLOR, zorder=5))

# 6. Sliding boundary (dashed)
ax.add_patch(plt.Circle((0, 0), p.R_sb, fill=False,
                         linestyle='--', edgecolor='#444444',
                         linewidth=0.9, zorder=6))

# 7-9. Rotor iron, magnets, air pockets
pole_centres = [(j + 0.5) * 360.0 / p.Qp for j in range(p.Qp)]

# 7. Rotor iron disk
ax.add_patch(plt.Circle((0, 0), p.R_ro, color='#5A7FBF', zorder=7))

# 8. Magnets drawn as rectangles using actual MotorParams geometry
#    R_mi = R_ro - h_bridge - h_m  (magnet inner),  R_mo = R_ro - h_bridge
#    Iron bridge (h_bridge thick) sits between R_mo and R_ro — visible in schematic.
hw = p.w_mag / 2
for j, pc_deg in enumerate(pole_centres):
    pc_rad = pc_deg * π / 180
    cos_c, sin_c = math.cos(pc_rad), math.sin(pc_rad)
    def _cart(r, t, cc=cos_c, ss=sin_c):
        return (r * cc - t * ss, r * ss + t * cc)
    corners = [_cart(p.R_mi, -hw), _cart(p.R_mi, +hw),
               _cart(p.R_mo, +hw), _cart(p.R_mo, -hw)]
    ax.add_patch(mpatches.Polygon(corners, closed=True, color=MAG_COLOR, zorder=8))
    r_lab = (p.R_mi + p.R_mo) / 2
    ax.text(r_lab * cos_c, r_lab * sin_c,
            'N' if j % 2 == 0 else 'S',
            ha='center', va='center', fontsize=6, color='white',
            fontweight='bold', zorder=9)

# 9. Air pockets as rectangles (flux barriers), flanking each magnet
if p.w_air > 0:
    wa = p.w_air
    for pc_deg in pole_centres:
        pc_rad = pc_deg * π / 180
        cos_c, sin_c = math.cos(pc_rad), math.sin(pc_rad)
        def _cart2(r, t, cc=cos_c, ss=sin_c):
            return (r * cc - t * ss, r * ss + t * cc)
        for side in (-1, +1):
            t_near = side * hw
            t_far  = side * (hw + wa)
            corners = [_cart2(p.R_mi, t_near), _cart2(p.R_mi, t_far),
                       _cart2(p.R_mo, t_far),  _cart2(p.R_mo, t_near)]
            ax.add_patch(mpatches.Polygon(corners, closed=True,
                                          color=POCKET_COLOR, zorder=9))
# 10. Shaft
ax.add_patch(plt.Circle((0, 0), p.R_ri, color=SHAFT_COLOR, zorder=10))

# ── Dimension annotations ──────────────────────────────────────────────────────
for r, lbl in [(p.R_so, 'R_so'), (p.R_si, 'R_si'), (p.R_ro, 'R_ro'),
               (p.R_mo, 'R_mo'), (p.R_mi, 'R_mi'), (p.R_ri, 'R_ri')]:
    ax.annotate('', xy=(r, 0), xytext=(0, 0),
                arrowprops=dict(arrowstyle='<->', color='k', lw=0.7))
    ax.text(r / 2, 2.5, f'{lbl}\n{r:.0f}mm',
            ha='center', va='bottom', fontsize=6.5, color='k')

# ── Legend ────────────────────────────────────────────────────────────────────
handles = [
    mpatches.Patch(color=IRON_COLOR,   label='Stator / rotor iron'),
    mpatches.Patch(color=AIRGAP_COLOR, label=f'Air gap (g = {p.g} mm)'),
    mpatches.Patch(color=MAG_COLOR,    label='PM magnet'),
    mpatches.Patch(color=POCKET_COLOR, label='Air pocket'),
    mpatches.Patch(color=SHAFT_COLOR,  label='Shaft'),
]
for ph in ('A+', 'A-', 'B+', 'B-', 'C+', 'C-'):
    handles.append(mpatches.Patch(color=PHASE_COLORS[ph], label=f'Phase {ph}'))
ax.legend(handles=handles, loc='upper right',
          bbox_to_anchor=(1.30, 1.02), fontsize=8, framealpha=0.9)

lim = p.R_so * 1.06
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_title(f'{p.Qs}s / {p.Qp}p / {p.m}-ph IPM — '
             f'OD {2*p.R_so:.0f} mm, bore {2*p.R_si:.0f} mm, '
             f'n = {p.rpm:.0f} rpm', fontsize=11, pad=10)

plt.tight_layout()
plt.savefig('motor_schematic.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Stator Sector Geometry & Mesh

`build_stator()` creates the 45° stator sector with rectangular slots and
individual hairpin conductors, then meshes it with gmsh.

In [ ]:
with tempfile.NamedTemporaryFile(suffix='.msh', delete=False) as f:
    stator_msh = Path(f.name)

print('Building stator mesh...')
build_stator(p, mesh_out=stator_msh)
xy_s, tris_s, segs_s = load_msh(stator_msh)
n_tri_s = sum(len(t) for t in tris_s.values())
print(f'  Nodes: {len(xy_s)},  Triangles: {n_tri_s},  Bodies: {len(tris_s)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: full stator sector ──────────────────────────────────────────────────
ax = axes[0]
ax.set_aspect('equal')
ax.set_title('Stator sector — full view (45°)')
fill_tris(ax, xy_s, tris_s)
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')

# Label boundary curves at their midpoints
bc_labels = ('Domain', 'Stator_Right', 'Stator_Left', 'SB_Stator')
for nm, seg in segs_s.items():
    if not any(bc in nm for bc in bc_labels):
        continue
    pts = xy_s[seg.flatten()]
    mid = pts.mean(axis=0)
    ax.text(mid[0], mid[1], nm.replace('_', '\n'),
            ha='center', fontsize=5.5, color='#220088',
            bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6))

# ── Right: single slot zoom ───────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_aspect('equal')
ax2.set_title('Slot detail — hairpins, liner & opening')
fill_tris(ax2, xy_s, tris_s, lw=0.2)

# Zoom into slot 2 (third slot, mid-sector)
θ_slot = (2 + 0.5) * p.sp
r_slot = p.R_si + p.h1 + p.h_slot / 2
cx = r_slot * math.cos(θ_slot)
cy = r_slot * math.sin(θ_slot)
margin = p.h_slot * 0.75
ax2.set_xlim(cx - margin, cx + margin)
ax2.set_ylim(cy - margin, cy + margin)
ax2.set_xlabel('x [mm]')
ax2.set_ylabel('y [mm]')

# Legend
leg_items = [
    mpatches.Patch(color=IRON_COLOR,  label='Stator iron'),
    mpatches.Patch(color=AIRGAP_COLOR,label='Airgap (stator side)'),
    mpatches.Patch(color=OPEN_COLOR,  label='Slot opening'),
    mpatches.Patch(color=INSUL_COLOR, label='Slot liner'),
]
for ph in ('A+', 'C-', 'B+'):
    leg_items.append(mpatches.Patch(color=PHASE_COLORS[ph],
                                    label=f'Hairpins {ph}'))
axes[0].legend(handles=leg_items, fontsize=7.5, loc='upper left')

# Annotate hairpin dimensions in zoom
ax2.annotate(
    f'b_hp={p.b_hp:.2f}mm\nh_hp={p.h_hp:.3f}mm\nn_hp={p.n_hp}',
    xy=(cx + margin * 0.4, cy + margin * 0.5),
    fontsize=7.5,
    bbox=dict(boxstyle='round', fc='white', alpha=0.85),
)

plt.tight_layout()
plt.savefig('stator_mesh.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Fill factor: {p.fill_factor*100:.1f}%  |  '
      f'Cu area/slot: {p.Carea*1e6:.2f} mm²')

## 4. Rotor Sector Geometry & Mesh

`build_rotor()` creates the 45° rotor sector with the inset permanent magnet,
flux-barrier air pockets, rotor iron, and shaft regions.

In [ ]:
with tempfile.NamedTemporaryFile(suffix='.msh', delete=False) as f:
    rotor_msh = Path(f.name)

print('Building rotor mesh...')
build_rotor(p, mesh_out=rotor_msh)
xy_r, tris_r, segs_r = load_msh(rotor_msh)
n_tri_r = sum(len(t) for t in tris_r.values())
print(f'  Nodes: {len(xy_r)},  Triangles: {n_tri_r},  Bodies: {len(tris_r)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Left: full rotor sector ───────────────────────────────────────────────────
ax = axes[0]
ax.set_aspect('equal')
ax.set_title('Rotor sector — full view (45°)')
fill_tris(ax, xy_r, tris_r)
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')
axes[0].legend(handles=[
    mpatches.Patch(color='#5A7FBF',   label='Rotor iron'),
    mpatches.Patch(color=MAG_COLOR,   label=f'PM magnet ({p.w_mag:.1f}×{p.h_m:.1f} mm)'),
    mpatches.Patch(color=POCKET_COLOR,label=f'Air pocket ({p.w_air:.1f} mm)'),
    mpatches.Patch(color=SHAFT_COLOR, label='Shaft'),
    mpatches.Patch(color=AIRGAP_COLOR,label='Airgap (rotor side)'),
], fontsize=7.5, loc='lower left')

# ── Right: magnet & air-pocket detail ────────────────────────────────────────
ax2 = axes[1]
ax2.set_aspect('equal')
ax2.set_title('Magnet & flux-barrier detail')
fill_tris(ax2, xy_r, tris_r, lw=0.2)

# Zoom into the right end of the magnet (magnet-to-air-pocket interface)
r_center = p.R_mi + p.h_m / 2
half_angle_rad = p.mag_frac * p.θs / 2
θ_right = p.θs / 2 + half_angle_rad     # right edge of magnet
cx2 = r_center * math.cos(θ_right)
cy2 = r_center * math.sin(θ_right)
margin2 = max(p.w_air * 4 + 2, p.h_m * 2)
ax2.set_xlim(cx2 - margin2, cx2 + margin2 * 0.7)
ax2.set_ylim(cy2 - margin2 * 0.7, cy2 + margin2)
ax2.set_xlabel('x [mm]')
ax2.set_ylabel('y [mm]')

# Dimension callout
ax2.annotate(
    f'h_m = {p.h_m} mm\nw_air = {p.w_air} mm\nB_r = {p.B_r} T',
    xy=(cx2 - margin2 * 0.5, cy2 + margin2 * 0.4),
    fontsize=8,
    bbox=dict(boxstyle='round', fc='white', alpha=0.85),
)

plt.tight_layout()
plt.savefig('rotor_mesh.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'R_ro = {p.R_ro} mm  |  R_mi = {p.R_mi} mm  |  '
      f'w_mag = {p.w_mag:.2f} mm  |  A_mag = {p.A_mag:.1f} mm²')

## 5. Full Sector Mesh & Airgap Detail

`build_mesh()` combines the stator and rotor into a single mesh.
The **sliding boundary** (SB) at R_sb uses *non-conforming* nodes:
`SB_Stator` (stator side) and `SB_Rotor` (rotor side) are the same geometric
circle but with independent mesh densities — Elmer couples them via
**Mortar boundary conditions**.

In [ ]:
with tempfile.NamedTemporaryFile(suffix='.msh', delete=False) as f:
    full_msh = Path(f.name)

print('Building full sector mesh...')
build_mesh(p, mesh_out=full_msh)
xy_f, tris_f, segs_f = load_msh(full_msh)
n_tri_f = sum(len(t) for t in tris_f.values())
print(f'  Nodes: {len(xy_f)},  Triangles: {n_tri_f},  Bodies: {len(tris_f)}')

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# ── Left: full 45° sector ─────────────────────────────────────────────────────
ax = axes[0]
ax.set_aspect('equal')
ax.set_title('Full 45° sector mesh (stator + rotor)')
fill_tris(ax, xy_f, tris_f)
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')

# Mark key radii
for r, lbl, col in [
    (p.R_so, 'R_so',  'k'),
    (p.R_si, 'R_si',  '#226622'),
    (p.R_sb, 'R_sb',  '#882222'),
    (p.R_ro, 'R_ro',  '#226622'),
    (p.R_ri, 'R_ri',  'k'),
]:
    # arc tick
    θ_tick = 0.04
    for sgn in (-1, +1):
        ax.plot([r * math.cos(sgn * θ_tick), r],
                [r * math.sin(sgn * θ_tick), 0],
                '-', color=col, lw=0.5, alpha=0.5)
    ax.text(r, -3.5, lbl, ha='center', va='top', fontsize=6.5, color=col)

axes[0].legend(handles=[
    mpatches.Patch(color=IRON_COLOR,   label='Stator iron'),
    mpatches.Patch(color=AIRGAP_COLOR, label='Airgap'),
    mpatches.Patch(color='#5A7FBF',    label='Rotor iron'),
    mpatches.Patch(color=MAG_COLOR,    label='Magnet'),
    mpatches.Patch(color=POCKET_COLOR, label='Air pocket'),
    mpatches.Patch(color=SHAFT_COLOR,  label='Shaft'),
    mpatches.Patch(color=PHASE_COLORS['A+'], label='Winding A+'),
    mpatches.Patch(color=PHASE_COLORS['C-'], label='Winding C-'),
    mpatches.Patch(color=PHASE_COLORS['B+'], label='Winding B+'),
], fontsize=7, loc='upper left')

# ── Right: airgap zoom (non-conforming SB) ────────────────────────────────────
ax2 = axes[1]
ax2.set_aspect('equal')
ax2.set_title(
    f'Airgap zoom — non-conforming sliding boundary\n'
    f'(g = {p.g} mm,  R_sb = {p.R_sb} mm,  lc_gap = {p.lc_gap} mm)'
)
fill_tris(ax2, xy_f, tris_f, lw=0.25)

# Highlight SB curves with distinct colours
sb_colors = {'SB_Stator': '#FF2222', 'SB_Rotor': '#2222FF'}
sb_nodes  = {}
for nm, seg in segs_f.items():
    for key, col in sb_colors.items():
        if key == nm:
            pts = xy_f[seg]
            for s in seg:
                ax2.plot(xy_f[s, 0], xy_f[s, 1], '-', color=col, lw=2.0, zorder=10)
            ax2.scatter(xy_f[seg.flatten(), 0], xy_f[seg.flatten(), 1],
                        s=8, color=col, zorder=11, linewidths=0)
            sb_nodes[key] = len(set(seg.flatten()))

# Zoom to airgap band near mid-sector
θ_mid = p.θs / 2
x_mid = p.R_sb * math.cos(θ_mid)
y_mid = p.R_sb * math.sin(θ_mid)
margin_gap = 5.0      # ±5 mm
ax2.set_xlim(x_mid - margin_gap, x_mid + margin_gap)
ax2.set_ylim(y_mid - margin_gap, y_mid + margin_gap)
ax2.set_xlabel('x [mm]')
ax2.set_ylabel('y [mm]')

ax2.legend(handles=[
    mpatches.Patch(color='#FF2222', label=f'SB_Stator ({sb_nodes.get("SB_Stator","?")!s} nodes)'),
    mpatches.Patch(color='#2222FF', label=f'SB_Rotor  ({sb_nodes.get("SB_Rotor","?")!s} nodes)'),
    mpatches.Patch(color=AIRGAP_COLOR, label='Airgap_Stator'),
    mpatches.Patch(color=AIRGAP_COLOR, label='Airgap_Rotor',
                   alpha=0.5),
], fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('full_mesh.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'SB nodes — Stator: {sb_nodes.get("SB_Stator","?")}  '
      f'Rotor: {sb_nodes.get("SB_Rotor","?")}  '
      f'(non-conforming → Mortar BC)')

## 6. Phase / Winding Assignment

The 45° sector contains **6 slots** (= Qs / Qp = 48 / 8).
The phase map `[A+, A+, C-, C-, B+, B+]` repeats with alternating polarity
across all 8 sectors to form the complete 3-phase winding.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: sector winding table ────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(-0.3, p.ns * 1.1 + 0.3)
ax.set_ylim(-1.2, p.n_hp * 1.2 + 1.0)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Winding layout — 45° sector  (slot × layer)')

W = 0.85   # slot box width
H = 0.85   # layer box height

# Draw hairpin grid
for i, ph in enumerate(_PHASE_MAP):
    # Slot label at top
    ax.text(i + W / 2, p.n_hp * H + 0.1, f'S{i}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    # Phase label below
    ax.text(i + W / 2, -0.65, ph,
            ha='center', va='center', fontsize=9,
            color=PHASE_COLORS[ph], fontweight='bold')
    for k in range(p.n_hp):
        rect = mpatches.FancyBboxPatch(
            (i + 0.05, k + 0.05), W - 0.10, H - 0.10,
            boxstyle='round,pad=0.04',
            facecolor=PHASE_COLORS[ph], edgecolor='k', lw=0.6)
        ax.add_patch(rect)
        ax.text(i + W / 2, k + H / 2, f'HP{k}',
                ha='center', va='center', fontsize=7, color='white')

# Axis labels
ax.text(-0.2, p.n_hp * H / 2, 'Layer (radial)',
        ha='center', va='center', fontsize=8, rotation=90)
ax.text(p.ns / 2, -1.1, 'Slot index',
        ha='center', va='top', fontsize=8)

# Layer labels (inner → outer)
for k in range(p.n_hp):
    ax.text(-0.25, k + H / 2, str(k),
            ha='right', va='center', fontsize=7)

# ── Right: full-machine winding wheel ─────────────────────────────────────────
ax2 = axes[1]
ax2.set_aspect('equal')
ax2.axis('off')
ax2.set_title(f'Full winding wheel ({p.Qs} slots, {p.Qp} poles)')

for i, ph in enumerate(full_phases):
    θ1 = i * slot_deg
    θ2 = θ1 + slot_deg * 0.82   # slight gap for clarity
    ax2.add_patch(Wedge((0, 0), 1.0, θ1, θ2,
                        width=0.38, color=PHASE_COLORS[ph], zorder=2))
    # Slot number label for first few
    if i < 8 or i >= p.Qs - 2:
        θ_mid_r = (θ1 + θ2) / 2 * π / 180
        r_lab = 0.84
        ax2.text(r_lab * math.cos(θ_mid_r), r_lab * math.sin(θ_mid_r),
                 str(i), ha='center', va='center', fontsize=5.5, color='white')

# Sector boundary ticks
for j in range(p.Qp):
    θ_rad = j * 2 * π / p.Qp
    ax2.plot([0, 1.06 * math.cos(θ_rad)], [0, 1.06 * math.sin(θ_rad)],
             'k--', lw=0.5, alpha=0.45, zorder=1)
    ax2.text(1.09 * math.cos(θ_rad + π / p.Qp),
             1.09 * math.sin(θ_rad + π / p.Qp),
             f'P{j}', ha='center', va='center', fontsize=7)

ax2.set_xlim(-1.25, 1.25)
ax2.set_ylim(-1.25, 1.25)

# Phase legend
ax2.legend(
    handles=[mpatches.Patch(color=PHASE_COLORS[ph], label=f'Phase {ph}')
             for ph in ('A+', 'A-', 'B+', 'B-', 'C+', 'C-')],
    fontsize=8, loc='upper right',
    bbox_to_anchor=(1.38, 1.0), ncol=2,
)

plt.tight_layout()
plt.savefig('phase_assignment.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Phase map (sector): {_PHASE_MAP}')
print(f'q = {p.q:.0f} slots/pole/phase,  ns = {p.ns} slots/sector')
print(f'Peak J = {p.J_peak/1e6:.2f} MA/m²  '
      f'(Is = {p.Is:.0f} A,  Carea = {p.Carea*1e6:.3f} mm²)')

## 7. FEA Model — SIF Generation

`gen_sif()` produces the Elmer Simulation Input File (SIF) from `MotorParams`.
The file encodes geometry, materials, boundary conditions, and solver settings
as MATC (Elmer's expression language) expressions.

In [ ]:
import importlib, gen_sif as _gen_sif_mod
importlib.reload(_gen_sif_mod)
from gen_sif import gen_sif

sif = gen_sif(p, mesh_dir='mesh', results_dir='results', suffix='2d')

n_bodies    = len(re.findall(r'^Body \d',           sif, re.MULTILINE))
n_bfs       = len(re.findall(r'^Body Force \d',     sif, re.MULTILINE))
n_materials = len(re.findall(r'^Material \d',       sif, re.MULTILINE))
n_solvers   = len(re.findall(r'^Solver \d',         sif, re.MULTILINE))
n_bcs       = len(re.findall(r'^Boundary Condition', sif, re.MULTILINE))
n_lines     = sif.count('\n')

print('Generated SIF structure')
print('=' * 40)
print(f'  Lines:               {n_lines}')
print(f'  Materials:           {n_materials}')
print(f'  Solvers:             {n_solvers}')
print(f'  Bodies:              {n_bodies}')
print(f'  Body forces:         {n_bfs}')
print(f'  Boundary conditions: {n_bcs}')

# Body numbering summary
print()
print('Body numbering:')
body_labels = (
    ['Stator_Iron', 'Airgap_Stator']
    + [f'S{i}_{n}'
       for i in range(p.ns)
       for n in ['Opening', 'Insul'] + [f'HP{k}' for k in range(p.n_hp)]]
    + ['Airgap_Rotor', 'Rotor_Iron', 'Magnet', 'AirPocket', 'Shaft']
)
for j, lbl in enumerate(body_labels[:14], start=1):
    print(f'  {j:3d}: {lbl}')
print(f'   ...  ({n_bodies} bodies total)')
print(f'  {len(body_labels):3d}: Shaft  [Rotor_Iron = {rotor_body_id(p)}]')

print()
print('Solver chain:')
solvers = [
    ('1', 'RigidMeshMapper',          'Rotor rotation at each timestep'),
    ('2', 'MagnetoDynamics2D',         'A-formulation magnetics (AV-source)'),
    ('3', 'MagnetoDynamicsCalcFields', 'B, H, J field extraction'),
    ('4', 'ResultOutputSolver',        'VTU export'),
    ('5', 'SaveScalars',               'Torque, energy, losses → scalars.dat'),
    ('6', 'FourierLossSolver',         'Hysteresis + eddy losses → loss-2d.dat'),
]
for num, name, desc in solvers:
    print(f'  Solver {num}: {name:<30s}  # {desc}')

print()
print('--- MATC parameter header (first 20 expression lines) ---')
matc_end = sif.index('Header')
for line in sif[:matc_end].split('\n')[:22]:
    if line.strip():
        print(line)

In [ ]:
# Visual overview of the SIF section lengths
sections = [
    ('MATC header',         sif[:sif.index('Header')].count('\n')),
    ('Header + Constants',  sif[sif.index('Header'):sif.index('Simulation')].count('\n')),
    ('Simulation',          sif[sif.index('Simulation'):sif.index('Material 1')].count('\n')),
    ('Materials',           sif[sif.index('Material 1'):sif.index('Body Force 1')].count('\n')),
    ('Body Forces',         sif[sif.index('Body Force 1'):sif.index('Body 1\n')].count('\n')),
    ('Bodies',              sif[sif.index('Body 1\n'):sif.index('Equation 1')].count('\n')),
    ('Equations + Solvers', sif[sif.index('Equation 1'):sif.index('Boundary Condition')].count('\n')),
    ('Boundary Conditions', sif[sif.index('Boundary Condition'):].count('\n')),
]

fig, ax = plt.subplots(figsize=(9, 4))
labels = [s[0] for s in sections]
values = [s[1] for s in sections]
bars = ax.barh(labels, values, color='#4472C4', edgecolor='k', lw=0.5)
for bar, v in zip(bars, values):
    ax.text(v + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{v} lines', va='center', fontsize=8)
ax.set_xlabel('Lines of SIF')
ax.set_title(f'SIF section sizes  (total: {n_lines} lines, {len(sif)//1024} kB)')
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('sif_structure.png', dpi=150, bbox_inches='tight')
plt.show()

## 7b. Run the Simulation

Write the generated SIF to disk and call `ElmerSolver`.  
The mesh must be in Elmer format under `mesh/`; if it is absent the cell builds
it with `gen_mesh.py` and converts it with `ElmerGrid 14 2`.

> **Runtime**: ~5–15 min on a laptop (241 timesteps, fine mesh). Run with `-np 4` for a 4× speedup via MPI (not shown here).

In [ ]:
import importlib, shutil, subprocess, time
import gen_sif as _gen_sif_mod
importlib.reload(_gen_sif_mod)
from gen_sif import gen_sif

SIF_FILE  = Path('case_nb.sif')
MESH_DIR  = Path('mesh')
RES_DIR   = Path('results')
CWD       = Path('.').resolve()

# ── 1. Generate SIF ───────────────────────────────────────────────────────────
sif = gen_sif(p, mesh_dir='mesh', results_dir='results', suffix='2d')
SIF_FILE.write_text(sif)
print(f'SIF written → {SIF_FILE}  ({SIF_FILE.stat().st_size:,} bytes)')

# ── 2. Build / verify Elmer mesh ──────────────────────────────────────────────
def _mesh_names_ok(mesh_dir):
    names_file = mesh_dir / 'mesh.names'
    if not names_file.exists():
        return False
    txt = names_file.read_text()
    return 'Stator_Iron' in txt and 'Rotor_Iron' in txt and 'Magnet' in txt

elmer_grid = shutil.which('ElmerGrid')
if elmer_grid is None:
    raise RuntimeError('ElmerGrid not found — install Elmer FEM.')

if _mesh_names_ok(MESH_DIR):
    print(f'Mesh OK: {MESH_DIR}/')
else:
    if MESH_DIR.exists():
        shutil.rmtree(MESH_DIR)
        print('Stale mesh detected — rebuilding...')
    else:
        print('Building mesh (gmsh) ...')
    gmsh_msh = MESH_DIR / 'motor.msh'
    MESH_DIR.mkdir(exist_ok=True)
    build_mesh(p, mesh_out=gmsh_msh)
    print('Converting mesh (ElmerGrid 14 2) ...')
    subprocess.run(
        [elmer_grid, '14', '2', str(gmsh_msh), '-autoclean', '-out', str(MESH_DIR)],
        cwd=CWD, check=True, capture_output=True,
    )
    if _mesh_names_ok(MESH_DIR):
        print('  Body names verified ✓')
    else:
        print('  WARNING: mesh body names unexpected')

RES_DIR.mkdir(exist_ok=True)

# ── 3. Run ElmerSolver (serial) ───────────────────────────────────────────────
solver = shutil.which('ElmerSolver')
if solver is None:
    raise RuntimeError('ElmerSolver not found — install Elmer FEM.')

cmd = [solver, str(SIF_FILE)]
print(f'\nCommand: {" ".join(cmd)}')
print('─' * 60)
t0 = time.perf_counter()
proc = subprocess.run(cmd, cwd=CWD, capture_output=True, text=True)
elapsed = time.perf_counter() - t0

lines = proc.stdout.strip().splitlines() if proc.stdout else []
tail  = lines[-50:] if len(lines) > 50 else lines
if len(lines) > 50:
    print(f'... ({len(lines) - 50} earlier lines omitted) ...')
for ln in tail:
    print(ln)
if proc.returncode != 0 and proc.stderr:
    print('\n--- stderr (last 40 lines) ---')
    for ln in proc.stderr.strip().splitlines()[-40:]:
        print(ln)
print('─' * 60)
status = 'SUCCESS' if proc.returncode == 0 else f'FAILED (rc={proc.returncode})'
print(f'Elapsed: {elapsed:.1f} s   Status: {status}')


## 7c. Current Angle (γ) Sweep

Sweep the current advance angle γ from −45° to +45° around the pure q-axis
(γ = 0°).  For an IPM motor the MTPA point lies at a negative γ (some
demagnetising *i*d that boosts reluctance torque).

Each point runs a full transient simulation (~3 min each → ~30 min total).

In [ ]:
import importlib, shutil, subprocess, time, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import gen_sif as _gen_sif_mod
importlib.reload(_gen_sif_mod)
from gen_sif import gen_sif
import postprocess as _pp_mod
importlib.reload(_pp_mod)
from postprocess import read_scalars

SWEEP_BASE = Path('results_sweep')
SIF_BASE   = Path('case_sweep')
SWEEP_BASE.mkdir(exist_ok=True)
SIF_BASE.mkdir(exist_ok=True)

solver = shutil.which('ElmerSolver')
if solver is None:
    print('ElmerSolver not found — install Elmer FEM to run the sweep.')
else:
    gammas   = np.linspace(-45, 45, 10)
    n_workers = min(len(gammas), os.cpu_count() or 4)
    print(f'Launching {len(gammas)} γ-angle runs with {n_workers} parallel workers...')

    def _run_one(args):
        idx, gamma = args
        res_dir = SWEEP_BASE / f'g{idx}'
        sif_path = SIF_BASE / f'g{idx}.sif'
        res_dir.mkdir(exist_ok=True)

        sif_txt = gen_sif(
            p,
            mesh_dir='mesh',
            results_dir=str(res_dir),
            suffix='sw',
            gamma_deg=float(gamma),
        )
        sif_path.write_text(sif_txt)

        t0 = time.perf_counter()
        proc = subprocess.run(
            [solver, str(sif_path)],
            cwd=Path('.').resolve(),
            capture_output=True, text=True,
        )
        elapsed = time.perf_counter() - t0
        return idx, gamma, elapsed, proc.returncode, res_dir

    results = {}   # idx → (gamma, T_mean, T_max, T_min)

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_run_one, (i, g)): i for i, g in enumerate(gammas)}
        for fut in as_completed(futures):
            idx, gamma, elapsed, rc, res_dir = fut.result()
            if rc != 0:
                print(f'  γ={gamma:+6.1f}°  FAILED  ({elapsed:.0f}s)')
                results[idx] = (gamma, float('nan'), float('nan'), float('nan'))
            else:
                sc = read_scalars(res_dir / 'scalars.dat', p)
                results[idx] = (gamma, sc['T_mean_Nm'], sc['T_max_Nm'], sc['T_min_Nm'])
                print(f'  γ={gamma:+6.1f}°  T_mean={sc["T_mean_Nm"]:7.2f} N·m  '
                      f'({elapsed:.0f}s  src={sc["torque_source"]})')

    # Sort by γ index for plotting
    sorted_res = [results[i] for i in range(len(gammas))]
    gammas_arr = np.array([r[0] for r in sorted_res])
    T_m        = np.array([r[1] for r in sorted_res])
    T_x        = np.array([r[2] for r in sorted_res])
    T_n        = np.array([r[3] for r in sorted_res])

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.fill_between(gammas_arr, T_n, T_x, alpha=0.25, label='Ripple band')
    ax.plot(gammas_arr, T_m, 'o-', lw=2, label='Mean torque')
    ax.axvline(0, color='gray', ls='--', lw=0.8, label='q-axis (γ=0)')
    if not np.all(np.isnan(T_m)):
        idx_max = int(np.nanargmax(T_m))
        ax.axvline(gammas_arr[idx_max], color='red', ls=':', lw=1.2,
                   label=f'MTPA  γ={gammas_arr[idx_max]:.1f}°')
    ax.set_xlabel('Current advance angle γ [deg]')
    ax.set_ylabel('Torque [N·m]')
    ax.set_title('Torque vs. current advance angle')
    ax.legend()
    fig.tight_layout()
    plt.show()


## 8. Postprocessing Results

`postprocess.py` extracts torque, power and iron losses from the Elmer output
files.  The cell below reads `results/scalars.dat` and `results/loss-2d.dat`
produced by a completed `ElmerSolver case.sif` run.

**Note:** If `results/` is not present the cell prints a warning and skips the plots.

In [ ]:
if not RESULTS_DIR.exists():
    print('WARNING: results/ directory not found.')
    print('Run  ElmerSolver case.sif  first, then re-execute this cell.')
else:
    m = summarise(RESULTS_DIR, p, suffix='2d', skip_cycles=1)
    print_summary(m)

In [ ]:
if not RESULTS_DIR.exists() or m.get('T_mean_Nm') is None:
    print('Skipping plots — no results available.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Left: torque waveform ─────────────────────────────────────────────────
    ax = axes[0]
    t_ms  = m['t_s'] * 1000.0
    T_Nm  = m['torque_Nm']
    fme   = p.rpm / 60.0
    T_mech_ms = 1000.0 / fme    # mechanical period in ms

    ax.plot(t_ms, T_Nm, 'b-', lw=1.0, label='Group-1 torque')
    if m.get('airgap_Nm') is not None:
        ax.plot(t_ms, m['airgap_Nm'], 'r--', lw=0.8, alpha=0.7,
                label='Air-gap torque')

    # Steady-state window highlight
    ax.axvspan(T_mech_ms, t_ms[-1], alpha=0.07, color='green')
    ax.axhline(m['T_mean_Nm'], color='#228822', lw=1.3, ls=':',
               label=f'Mean = {m["T_mean_Nm"]:.1f} N·m')

    # Ripple band
    ax.axhline(m['T_max_Nm'], color='#882222', lw=0.7, ls='--', alpha=0.5)
    ax.axhline(m['T_min_Nm'], color='#882222', lw=0.7, ls='--', alpha=0.5)
    ax.fill_between(t_ms, m['T_min_Nm'], m['T_max_Nm'],
                    alpha=0.06, color='#882222')

    ax.set_xlabel('Time [ms]')
    ax.set_ylabel('Torque [N·m]')
    ax.set_title('Torque waveform (full machine, SCALE applied)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    ax.annotate(
        f'Ripple: {m["T_ripple_pct"]:.1f}%\n'
        f'P_mech: {m["P_mech_kW"]:.2f} kW\n'
        f'n = {p.rpm:.0f} rpm',
        xy=(0.97, 0.05), xycoords='axes fraction', ha='right',
        fontsize=8.5,
        bbox=dict(boxstyle='round', fc='white', alpha=0.85),
    )
    # Period annotation
    ax.annotate('', xy=(T_mech_ms, m['T_mean_Nm'] * 0.92),
                xytext=(0, m['T_mean_Nm'] * 0.92),
                arrowprops=dict(arrowstyle='<->', color='#555555', lw=0.8))
    ax.text(T_mech_ms / 2, m['T_mean_Nm'] * 0.905,
            f'cycle 1\n({T_mech_ms:.1f} ms)',
            ha='center', va='top', fontsize=7, color='#555555')

    # ── Right: iron loss breakdown ────────────────────────────────────────────
    ax2 = axes[1]

    s_h = m.get('stator_hyst_W') or 0.0
    s_e = m.get('stator_eddy_W') or 0.0
    r_h = m.get('rotor_hyst_W')  or 0.0
    r_e = m.get('rotor_eddy_W')  or 0.0

    cats   = ['Stator\nhyst.', 'Stator\neddy', 'Rotor\nhyst.', 'Rotor\neddy']
    vals   = [s_h, s_e, r_h, r_e]
    colors = [IRON_COLOR, IRON_COLOR, '#5A7FBF', '#5A7FBF']
    hatches= ['', '////', '', '\\\\\\\\']

    bars = ax2.bar(cats, vals, color=colors, edgecolor='k', lw=0.5)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    for bar, v in zip(bars, vals):
        if v > 0.1:
            ax2.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + max(vals) * 0.01,
                     f'{v:.1f} W',
                     ha='center', va='bottom', fontsize=8.5)

    total = m.get('total_W') or 0.0
    eta   = m.get('efficiency_pct') or 0.0
    ax2.set_title(
        f'Iron losses — total {total:.1f} W\n'
        f'(indicative η ≥ {eta:.1f}%,  SCALE = {p.SCALE:.4f} m)'
    )
    ax2.set_ylabel('Power [W]')
    ax2.grid(True, axis='y', alpha=0.3)

    ax2.legend(handles=[
        mpatches.Patch(color=IRON_COLOR, label='Stator iron'),
        mpatches.Patch(color='#5A7FBF',  label='Rotor iron'),
        mpatches.Patch(color='white', edgecolor='k',
                       hatch='////', label='Eddy current'),
        mpatches.Patch(color='white', edgecolor='k',
                       label='Hysteresis'),
    ], fontsize=8)

    plt.tight_layout()
    plt.savefig('postprocess_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Summary table
    print(f'\n  SCALE  = {p.SCALE:.4f} m  (Qp={p.Qp} × L={p.L_active} mm)')
    print(f'  n_steps = {m["n_steps"]}  '
          f'(nper=120, ncycle=2,  dt={1/(fme*120)*1000:.4f} ms)')

## 8b. Magnetic Flux Density Map

Plot **|B|** from the last VTU file of the most recent simulation.  `plot_flux_density()` reads Elmer's binary VTU output and uses `pyvista` (or `meshio` as fallback) to render a filled contour map of the flux density across the 45° sector.

In [ ]:
import importlib, glob, os
import matplotlib.pyplot as plt
import postprocess as _pp_mod
importlib.reload(_pp_mod)
from postprocess import plot_flux_density

RESULTS_DIR = Path('results')

# Pick the last VTU file written in the results directory
vtu_pattern = str(RESULTS_DIR / 'step-2d_t*.vtu')
vtu_files   = sorted(glob.glob(vtu_pattern))

if not vtu_files:
    print(f'No VTU files found matching {vtu_pattern}')
    print('Run section 7b first to generate simulation results.')
else:
    last_vtu = vtu_files[-1]
    print(f'Plotting: {os.path.basename(last_vtu)}')
    fig, ax = plot_flux_density(last_vtu, figsize=(7, 7))
    plt.show()

---
## Summary

| Step | Script | Output |
|------|--------|--------|
| 1 | `motor_params.py` | `MotorParams` dataclass with validated geometry |
| 2 | `gen_stator.py` | Stator sector `.msh` (slots, hairpins, airgap) |
| 3 | `gen_rotor.py` | Rotor sector `.msh` (magnets, pockets, shaft) |
| 4 | `gen_mesh.py` | Combined 45° sector `.msh` + optional Elmer format |
| 5 | `gen_sif.py` | `case.sif` (materials, bodies, solvers, BCs) |
| 6 | `ElmerSolver` | `scalars.dat`, `loss-2d.dat`, `*.vtu` |
| 7 | `postprocess.py` | Torque, power, iron losses, efficiency |

All steps are driven by a single `MotorParams` instance — change any parameter
and re-run to get a fully consistent mesh + FEA model.